# 09 Train RL Overlay SAC v1

This notebook calls `scripts/train_rl_overlay_sac_v1.py` to train, validate, and test the SAC overlay for monthly portfolio allocation. The notebook is intentionally thin: all production logic lives in the Python script.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
SCRIPT = PROJECT_ROOT / "scripts/train_rl_overlay_sac_v1.py"

print(f"Python executable: {sys.executable}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Script: {SCRIPT}")

## Configuration

Edit the parameters below before launching a full training run. For a quick smoke test, reduce `TOTAL_TIMESTEPS` and narrow the date ranges.

In [ ]:
OUTDIR = "data/rl_overlay_sac"

TRAIN_START = "2006-01"
TRAIN_END = "2016-12"
VAL_START = "2017-01"
VAL_END = "2019-12"
TEST_START = "2020-01"
TEST_END = "2025-12"

TOTAL_TIMESTEPS = 200_000
EVAL_FREQUENCY = 5_000
COST_BPS = 10.0
MAX_WEIGHT = None
SOLVER = "CLARABEL"
SEED = 42

USE_VECNORMALIZE = True
WARM_START_CVXPY = True

FF_FILE = None

## Preflight Checks

In [ ]:
required_paths = [
    SCRIPT,
    PROJECT_ROOT / "data/prediction/fm_oos_predictions.parquet",
    PROJECT_ROOT / "data/risk/risk_cov_npz",
    PROJECT_ROOT / "data/risk/risk_monthly_metadata.csv",
    PROJECT_ROOT / "data/panel/monthly_stock_panel_basic_full.parquet",
]

missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required paths:\n" + "\n".join(str(path) for path in missing)
    )

print("All required paths found.")

## Build Command

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT),
    "--project-root",
    str(PROJECT_ROOT),
    "--outdir",
    OUTDIR,
    "--train-start",
    TRAIN_START,
    "--train-end",
    TRAIN_END,
    "--val-start",
    VAL_START,
    "--val-end",
    VAL_END,
    "--test-start",
    TEST_START,
    "--test-end",
    TEST_END,
    "--total-timesteps",
    str(TOTAL_TIMESTEPS),
    "--eval-frequency",
    str(EVAL_FREQUENCY),
    "--cost-bps",
    str(COST_BPS),
    "--solver",
    SOLVER,
    "--seed",
    str(SEED),
    "--use-vecnormalize",
    str(USE_VECNORMALIZE).lower(),
    "--warm-start-cvxpy",
    str(WARM_START_CVXPY).lower(),
]

if MAX_WEIGHT is not None:
    cmd.extend(["--max-weight", str(MAX_WEIGHT)])

if FF_FILE is not None:
    cmd.extend(["--ff-file", str(FF_FILE)])

print(" ".join(shlex.quote(part) for part in cmd))

## Run Training Pipeline

In [ ]:
result = subprocess.run(cmd, cwd=PROJECT_ROOT, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Training script failed with return code {result.returncode}.")

## Inspect Outputs

In [ ]:
outdir = PROJECT_ROOT / OUTDIR
expected_outputs = [
    outdir / "models/best_model.zip",
    outdir / "models/final_model.zip",
    outdir / "train_history.csv",
    outdir / "validation_metrics.csv",
    outdir / "test_backtest.csv",
    outdir / "test_weights.parquet",
    outdir / "test_action_history.csv",
    outdir / "test_summary.txt",
    outdir / "test_cumret.png",
    outdir / "test_turnover.png",
    outdir / "test_lambda_timeseries.png",
    outdir / "test_tau_timeseries.png",
]

for path in expected_outputs:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:8s} {path}")

summary_path = outdir / "test_summary.txt"
if summary_path.exists():
    print("\n" + summary_path.read_text(encoding="utf-8"))